# 期現套利回測報告：可重跑分析 Notebook

## tl;dr

這個 Notebook 只讀取一月至七月已保存的 HBT CSV，不重跑回測。執行後會顯示排除異常日期後的 ROI、Sharpe proxy、損益組成、資金占用與資料品質檢查，並輸出專業報告使用的稽核表。

## 分析背景與方法

### 重要假設

- 資金周轉 ROI 沿用既有現金報表口徑：總損益（含未平倉鎖定 PnL）除以累計股票進場現金；不含期貨保證金，也不是固定本金報酬。
- Sharpe proxy 使用 active-day ROI、252 日年化、無風險利率 0；不是連續投資組合權益曲線的標準 Sharpe。
- 排除日期與 `replot_filtered_results.ipynb` 一致。

In [1]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import Markdown, display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / 'future_spot').exists():
    WORKSPACE_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == 'notebooks' and CURRENT_DIR.parent.name == 'future_spot':
    WORKSPACE_ROOT = CURRENT_DIR.parents[1]
else:
    raise FileNotFoundError('請從 repository root 或 future_spot/notebooks 執行')
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import future_spot.arbitrage.backtest_report as backtest_report
importlib.reload(backtest_report)
from future_spot.arbitrage.backtest_report import build_backtest_report
from future_spot.arbitrage.result_replot import PlotInterval

## 資料

### 1. 設定既有輸出與排除日期

In [2]:
OUTPUT_DIRS = [
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260101_20260131_latency_10ms',
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260201_20260228_latency_10ms',
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260301_20260331_latency_10ms',
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260401_20260430_latency_10ms',
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260501_20260531_latency_10ms',
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260521_20260526_latency_10ms',  # 六月 summary
    WORKSPACE_ROOT / 'future_spot/output/hbt_daily_full_market_20260701_20260731_latency_10ms',
]

REPORT_INTERVALS = [
    PlotInterval('2026-01', '2026-01-01', '2026-01-31'),
    PlotInterval('2026-02', '2026-02-01', '2026-02-28'),
    PlotInterval('2026-03', '2026-03-01', '2026-03-31'),
    PlotInterval('2026-04', '2026-04-01', '2026-04-30', ('2026-04-23',)),
    PlotInterval('2026-05', '2026-05-01', '2026-05-31', ('2026-05-04', '2026-05-19')),
    PlotInterval('2026-06', '2026-06-01', '2026-06-30', ('2026-06-24',)),
    PlotInterval('2026-07', '2026-07-01', '2026-07-31', ('2026-07-01', '2026-07-22', '2026-07-27')),
]

REPORT_OUTPUT_DIR = WORKSPACE_ROOT / 'future_spot/output/backtest_report_202601_202607'

### 2. 建立報告資料與稽核證據

In [3]:
report = build_backtest_report(OUTPUT_DIRS, REPORT_INTERVALS, REPORT_OUTPUT_DIR)
h = report.headline
display(Markdown(
    f"**過濾後結果：** 含未平倉損益的總 PnL 為 NT${h['total_pnl_including_open']/1_000_000:.2f}M，"
    f"資金周轉 ROI 為 {h['total_roi']:.3%}，Sharpe proxy 為 {h['sharpe_proxy']:.2f}。"
    f"未平倉鎖定 PnL 占 {h['open_pnl_share']:.1%}；可信度：**可分享，但須附帶限制說明**。"
))
headline_labels = {
    'period_start': '資料起日', 'period_end': '資料迄日',
    'observed_trade_days': '觀測交易日', 'active_roi_days': '有效 ROI 日',
    'entry_cash_out': '累計進場現金', 'realized_pnl': '已實現 PnL',
    'open_locked_pnl': '未平倉鎖定 PnL', 'total_pnl_including_open': '總 PnL（含未平倉）',
    'total_roi': '資金周轉 ROI', 'realized_roi': '已實現 ROI',
    'open_pnl_share': '未平倉 PnL 占比', 'sharpe_proxy': 'Sharpe proxy',
    'sortino_proxy': 'Sortino proxy', 'active_day_win_rate': 'active-day 勝率',
    'max_drawdown_twd': '最大回撤（NT$）', 'entries': '進場次數',
    'exits': '出場次數', 'open_pair_runs': '未平倉 pair-run',
    'completion_rate': '完成率', 'peak_daily_entry_cash': '單日進場現金峰值',
    'peak_daily_stuck_cash': '單日卡住現金峰值', 'summary_realized_pnl': 'Summary 已實現 PnL',
    'excluded_dates': '排除日期數', 'excluded_summary_pnl': '排除日期的 Summary PnL',
    'config_error_days': '設定錯誤日數', 'top_two_month_pnl_share': '前兩個月 PnL 占比',
}
headline_table = report.frame('headline_metrics').T.rename(columns={0: '數值'}).rename(index=headline_labels)
display(headline_table)

**過濾後結果：** 含未平倉損益的總 PnL 為 NT$17.73M，資金周轉 ROI 為 0.811%，Sharpe proxy 為 24.66。未平倉鎖定 PnL 占 73.2%；可信度：**可分享，但須附帶限制說明**。

,數值
資料起日,2026-01-08
資料迄日,2026-07-31
觀測交易日,125
有效 ROI 日,73
累計進場現金,2185380318.9713
已實現 PnL,4745406.31627
未平倉鎖定 PnL,12985068.9334
總 PnL（含未平倉）,17730475.24967
資金周轉 ROI,0.008113
已實現 ROI,0.002171


## 分析結果

### 3. 月度績效與資金占用

In [4]:
monthly_columns = [
    'month', 'observed_trade_days', 'active_days', 'entry_cash_out', 'realized_pnl',
    'open_locked_pnl', 'total_pnl_including_open', 'total_roi', 'sharpe_proxy',
    'completion_rate',
]
monthly_labels = {
    'month': '月份', 'observed_trade_days': '觀測交易日', 'active_days': '有效 ROI 日',
    'entry_cash_out': '進場現金', 'realized_pnl': '已實現 PnL',
    'open_locked_pnl': '未平倉鎖定 PnL', 'total_pnl_including_open': '總 PnL',
    'total_roi': '資金周轉 ROI', 'sharpe_proxy': 'Sharpe proxy', 'completion_rate': '完成率',
}
symbol_labels = {
    'spot_symbol': '現貨代號', 'pair_runs': 'pair-run 數', 'entry_cash_out': '進場現金',
    'realized_pnl': '已實現 PnL', 'open_locked_pnl': '未平倉鎖定 PnL',
    'total_pnl_including_open': '總 PnL', 'open_pairs': '未平倉 pair-run',
    'total_roi': '資金周轉 ROI', 'open_pnl_share': '未平倉 PnL 占比',
}
display(report.frame('monthly_performance')[monthly_columns].rename(columns=monthly_labels))
display(report.frame('symbol_performance').head(15).rename(columns=symbol_labels))

,月份,觀測交易日,有效 ROI 日,進場現金,已實現 PnL,未平倉鎖定 PnL,總 PnL,資金周轉 ROI,Sharpe proxy,完成率
0,2026-01,17,9,3.208240e+07,1.527114e+05,4.139351e+04,1.941049e+05,0.006050,44.400508,0.826923
1,2026-02,11,5,1.565104e+07,9.343417e+04,7.223632e+03,1.006578e+05,0.006431,23.220033,0.895833
2,2026-03,22,15,9.171978e+07,5.574185e+05,8.754107e+04,6.449595e+05,0.007032,28.842853,0.807074
3,2026-04,19,12,1.054213e+08,4.298182e+05,4.421772e+05,8.719954e+05,0.008272,15.802232,0.560000
4,2026-05,18,15,1.256352e+09,2.455307e+06,7.575020e+06,1.003033e+07,0.007984,65.460274,0.450265
5,2026-06,20,15,6.806822e+08,9.852488e+05,4.831215e+06,5.816464e+06,0.008545,38.510723,0.376768
6,2026-07,18,2,3.471284e+06,7.146802e+04,4.981737e+02,7.196619e+04,0.020732,196.997089,0.909091


,現貨代號,進場現金,已實現 PnL,未平倉鎖定 PnL,總 PnL,entries,exits,未平倉 pair-run,資金周轉 ROI,未平倉 PnL 占比
0,2408,2.002479e+08,518369.6730,788478.295,1.306848e+06,293,182.0,111.0,0.006526,0.603344
1,3037,1.880040e+08,674868.8860,608544.640,1.283414e+06,111,91.0,20.0,0.006827,0.474161
2,2454,9.006592e+07,0.0000,974089.230,9.740892e+05,10,0.0,10.0,0.010815,1.000000
3,3189,1.245347e+08,381788.5420,588803.526,9.705921e+05,125,85.0,40.0,0.007794,0.606644
4,4958,1.124799e+08,270485.3650,557201.612,8.276870e+05,110,60.0,50.0,0.007359,0.673203
5,3711,7.752292e+07,0.0000,800513.352,8.005134e+05,60,0.0,60.0,0.010326,1.000000
6,8046,8.583023e+07,85735.5880,679040.642,7.647762e+05,49,9.0,40.0,0.008910,0.887895
7,2337,8.015517e+07,337509.9974,266668.687,6.041787e+05,250,193.0,57.0,0.007538,0.441374
8,6147,6.958575e+07,197131.1720,402149.808,5.992810e+05,133,61.0,72.0,0.008612,0.671054
9,2449,6.102734e+07,80876.0777,458246.610,5.391227e+05,94,24.0,70.0,0.008834,0.849986


### 4. 資料品質與排除日期稽核

In [5]:
coverage_labels = {
    'interval': '月份', 'attempted_trade_days': '嘗試交易日', 'config_success_days': '設定成功日',
    'config_error_days': '設定錯誤日', 'config_error_dates': '錯誤日期',
    'summary_days_before_exclusion': '排除前 summary 日', 'excluded_days_found': '排除日',
    'included_summary_days': '納入 summary 日', 'active_roi_days': '有效 ROI 日',
}
excluded_labels = {
    'interval': '月份', 'trade_date': '日期', 'found_in_summary': 'summary 中可找到',
    'summary_realized_pnl_removed': '移除的 summary PnL', 'filled_pairs_removed': '移除的成交配對',
    'second_leg_failures_removed': '移除的第二腿失敗', 'reason': '原因',
}
display(report.frame('validation_checks'))
display(report.frame('coverage').rename(columns=coverage_labels))
display(report.frame('excluded_dates').rename(columns=excluded_labels))
print(f'報告稽核資料目錄：{report.output_dir}')
print(f'標準報告 artifact：{report.output_dir / "artifact.json"}')

,檢查項目,狀態,證據,重要性
0,Summary run_key 唯一性,通過,重複鍵數：0,高
1,Summary PnL 完整性,通過,PnL 空值列數：0,高
2,ROI 公式核對,通過,最大絕對差：1.01e-16,高
3,月度加總與總計核對,通過,月度總 PnL 已與過濾後的每日 ROI 加總一致,高
4,指定排除日期可找到,通過,找到 7/7 個排除日期,中
5,設定檔覆蓋率,注意,7 個嘗試交易日發生設定錯誤,高
6,未平倉部位依賴,注意,總 PnL 含未平倉鎖定損益，並非完全已實現的權益曲線,高
7,Sharpe 報酬口徑,注意,Sharpe 使用 active-day 的累計進場現金 ROI，不是固定本金投資組合報酬,高


,月份,嘗試交易日,設定成功日,設定錯誤日,錯誤日期,排除前 summary 日,排除日,納入 summary 日,有效 ROI 日
0,2026-01,21,17,4,"2026-01-02, 2026-01-05, 2026-01-06, 2026-01-07",17,0,17,9
1,2026-02,12,11,1,2026-02-23,11,0,11,5
2,2026-03,22,22,0,,22,0,22,15
3,2026-04,20,20,0,,20,1,19,12
4,2026-05,20,20,0,,20,2,18,15
5,2026-06,21,21,0,,21,1,20,15
6,2026-07,23,21,2,"2026-07-10, 2026-07-13",21,3,18,2


,月份,日期,summary 中可找到,移除的 summary PnL,移除的成交配對,移除的第二腿失敗,原因
0,2026-04,2026-04-23,True,-1.891843e+07,166,11057,使用者指定排除缺 tick 異常日期
1,2026-05,2026-05-04,True,-3.963816e+07,16,40897,使用者指定排除缺 tick 異常日期
2,2026-05,2026-05-19,True,-5.678798e+06,8,5249,使用者指定排除缺 tick 異常日期
3,2026-06,2026-06-24,True,-3.244169e+07,52,17327,使用者指定排除缺 tick 異常日期
4,2026-07,2026-07-01,True,-1.001800e+06,0,1787,使用者指定排除缺 tick 異常日期
5,2026-07,2026-07-22,True,-1.049600e+06,0,1502,使用者指定排除缺 tick 異常日期
6,2026-07,2026-07-27,True,-8.424000e+05,0,392,使用者指定排除缺 tick 異常日期


報告稽核資料目錄：/home/zoufuc/hftbacktest/future_spot/output/backtest_report_202601_202607
標準報告 artifact：/home/zoufuc/hftbacktest/future_spot/output/backtest_report_202601_202607/artifact.json


## 結論與後續行動

- ROI 與 Sharpe 看起來很強，但主要限制不是日內虧損，而是大量未平倉 pair-run、資金卡住，以及非固定本金的報酬口徑。
- 對外分享時必須同時呈現已實現 PnL、未平倉鎖定 PnL、完成率、資料排除與缺失日期。
- 下一步應建立固定本金、每日市值評價、納入期貨保證金與強制平倉的連續權益曲線。